# Service Design Processes Generator


## There are two service Demos one is a notary office (size 7) and the other is a hospital operation (size 15)

__Select and run either the first or the second cell__

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx

# Reset random seed for reproducibility
np.random.seed(2025)

# ======================================================
# 1. CORE REGISTRIES (7-NODE PINPOINT CONFIG)
# ======================================================
TOUCHPOINTS = [
    "Client_Intake",          # 0: Input (Linear)
    "Document_Triage",        # 1: Input (Linear)
    "Identity_Credentialing", # 2: HUB (High Risk / Fraud Vector)
    "Legal_Scrutiny",         # 3: Core (Clause Analysis)
    "The_Notarial_Act",       # 4: Core (Execution)
    "Compliance_Audit",       # 5: HUB (System of Record)
    "Journal_Archiving"       # 6: Output (Linear)
]

# METRIC KEYS (Aligns with Touchpoints for Node-Level Scoring)
METRIC_KEYS = TOUCHPOINTS

# ======================================================
# 2. FEATURE REGISTRY (FLAT LIST OF MATRIX COLUMNS)
# ======================================================
# This list corresponds exactly to the columns in the generated DATA_MATRIX
FEATURE_KEYS = [
    # 0: Client_Intake
    'wait_time_mins', 'client_sentiment',
    # 1: Document_Triage
    'page_count', 'doc_complexity_score',
    # 2: Identity_Credentialing (HUB)
    'biometric_confidence', 'fraud_risk_flag',
    # 3: Legal_Scrutiny
    'clauses_flagged', 'readability_index',
    # 4: The_Notarial_Act
    'ceremony_duration_sec', 'stamp_clarity_score',
    # 5: Compliance_Audit (HUB)
    'violation_severity', 'data_integrity_score',
    # 6: Journal_Archiving
    'upload_latency_ms', 'chain_hash_valid'
]

# Mapping specific features to their parent Node/Metric
metric_feature_map = {
    "Client_Intake":          ['wait_time_mins', 'client_sentiment'],
    "Document_Triage":        ['page_count', 'doc_complexity_score'],
    "Identity_Credentialing": ['biometric_confidence', 'fraud_risk_flag'],
    "Legal_Scrutiny":         ['clauses_flagged', 'readability_index'],
    "The_Notarial_Act":       ['ceremony_duration_sec', 'stamp_clarity_score'],
    "Compliance_Audit":       ['violation_severity', 'data_integrity_score'],
    "Journal_Archiving":      ['upload_latency_ms', 'chain_hash_valid']
}

# ======================================================
# 3. INTERACTION MATRIX & FORMULAS (7x7 KNOT LOGIC)
# ======================================================

# 7x7 DSM (Dependency Structure Matrix)
# Defines the critical rework loops:
# - Identity (2) -> Legal (3) -> Act (4)
# - Audit (5) -> Feedback Loop to Identity (2) & Legal (3)
METRIC_TARGET = np.array([
    #0  1  2  3  4  5  6
    [1, 1, 0, 0, 0, 0, 0], # 0: Intake -> Triage
    [0, 1, 1, 1, 0, 0, 0], # 1: Triage -> ID, Legal
    [0, 0, 1, 1, 1, 0, 0], # 2: ID -> Legal, Act (CRITICAL PATH)
    [0, 0, 0, 1, 1, 0, 0], # 3: Legal -> Act
    [0, 0, 0, 0, 1, 1, 0], # 4: Act -> Audit
    [0, 0, 1, 1, 0, 1, 1], # 5: Audit -> FEEDBACK LOOP (ID, Legal), Archive
    [0, 0, 0, 0, 0, 0, 1]  # 6: Archive
])

METRIC_TARGE1T = np.array([
    #0  1  2  3  4  5  6
    [1, 1, 0, 0, 1, 0, 1], # 0: Intake -> Triage
    [0, 1, 1, 1, 0, 1, 0], # 1: Triage -> ID, Legal
    [0, 0, 1, 1, 1, 0, 1], # 2: ID -> Legal, Act (CRITICAL PATH)
    [0, 0, 0, 1, 1, 0, 0], # 3: Legal -> Act
    [1, 0, 0, 0, 1, 1, 0], # 4: Act -> Audit
    [0, 0, 1, 1, 0, 1, 1], # 5: Audit -> FEEDBACK LOOP (ID, Legal), Archive
    [1, 1, 0, 0, 0, 0, 1]  # 6: Archive
])

# Scoring Formulas (Transfer Functions 0.0 - 1.0)
METRIC_FORMULAS = [
    lambda x: np.exp(-x / 15.0),           # 0: Intake (Minimize wait)
    lambda x: 1.0 / (1.0 + x),             # 1: Triage (Handle complexity)
    lambda x: x**3,                        # 2: Identity (Confidence must be high)
    lambda x: np.exp(-x * 0.5),            # 3: Legal (Minimize flagged clauses)
    lambda x: np.exp(-(x-300)**2 / 1e4),   # 4: Act (Target 5 min duration)
    lambda x: 1.0 / (1.0 + x*2),           # 5: Audit (Zero violations target)
    lambda x: 1.0 / (1.0 + x/200)          # 6: Archive (Minimize latency)
]

METRIC_FORMULAS1 = [
    lambda x: np.exp(-x / 15.0),           # 0: Intake (Minimize wait)
    lambda x: 1.0 / (1.0 + x),             # 1: Triage (Handle complexity)
    lambda x: x**3,                        # 2: Identity (Confidence must be high)
    lambda x: np.exp(-x * 0.5),            # 3: Legal (Minimize flagged clauses)
    lambda x: np.exp(-(x-300)**2 / 1e4),   # 4: Act (Target 5 min duration)
    lambda x: x*2,           # 5: Audit (Zero violations target)
    lambda x: x/200          # 6: Archive (Minimize latency)
]

# ======================================================
# 4. DIMENSIONALITY CONFIGURATION
# ======================================================
# 16 Dims for Hubs (ID, Audit)
# 12 Dims for Core (Legal, Act)
# 4-8 Dims for Linear
candidate_dims = [
    [4],  # 0: Client_Intake
    [8],  # 1: Document_Triage
    [16], # 2: Identity_Credentialing (HUB)
    [12], # 3: Legal_Scrutiny
    [12], # 4: The_Notarial_Act
    [16], # 5: Compliance_Audit (HUB)
    [4]   # 6: Journal_Archiving
]

D_graph = len(candidate_dims)

# ======================================================
# 5. GENERATOR ENGINE
# ======================================================
GENERATOR_MAP = {
    "Client_Intake": lambda: {
        "wait_time_mins": np.random.exponential(12),
        "client_sentiment": np.random.beta(5, 2)
    },
    "Document_Triage": lambda: {
        "page_count": np.random.randint(1, 50),
        "doc_complexity_score": np.random.choice([0.2, 0.5, 0.9])
    },
    "Identity_Credentialing": lambda: {
        "biometric_confidence": np.random.uniform(0.85, 0.999),
        "fraud_risk_flag": np.random.binomial(1, 0.02)
    },
    "Legal_Scrutiny": lambda: {
        "clauses_flagged": np.random.poisson(2),
        "readability_index": np.random.uniform(30, 80)
    },
    "The_Notarial_Act": lambda: {
        "ceremony_duration_sec": np.random.normal(300, 60),
        "stamp_clarity_score": np.random.uniform(0.9, 1.0)
    },
    "Compliance_Audit": lambda: {
        "violation_severity": np.random.exponential(0.5),
        "data_integrity_score": np.random.beta(9, 1)
    },
    "Journal_Archiving": lambda: {
        "upload_latency_ms": np.random.uniform(100, 500),
        "chain_hash_valid": 1.0
    }
}

# ======================================================
# 6. MATRIX BUILDER
# ======================================================
def build_data_matrix(num_samples=200):
    feature_list = []
    for tp in TOUCHPOINTS:
        samples = [GENERATOR_MAP[tp]() for _ in range(num_samples)]
        df = pd.DataFrame(samples)
        feature_list.append(df)

    raw_matrix = pd.concat(feature_list, axis=1).to_numpy()

    # Robust Normalization
    ptp = np.ptp(raw_matrix, axis=0)
    ptp[ptp == 0] = 1.0
    norm_matrix = (raw_matrix - raw_matrix.min(axis=0)) / ptp
    return norm_matrix

DATA_MATRIX = build_data_matrix()

# ======================================================
# 7. SYNTHETIC TARGET GENERATOR (HUB/KNOT LOGIC)
# ======================================================
def generate_notary_targets(DATA_MATRIX, candidate_dims):
    dims_flat = [i[0] for i in candidate_dims]
    num_nodes = len(dims_flat)
    targets = []
    current_col = 0



    print(f"{'ID':<3} | {'Touchpoint Name':<24} | {'Role':<12} | {'Dims'} | {'Signature Sample'}")
    print("-" * 85)

    for node_idx in range(num_nodes):
        dim_required = dims_flat[node_idx]

        # Row selection & Feature Slicing
        row = DATA_MATRIX[node_idx % DATA_MATRIX.shape[0]]
        block = row[current_col : current_col + dim_required]

        # REFLECTIVE PADDING
        if len(block) < dim_required:
            pad_width = dim_required - len(block)
            block = np.pad(block, (0, pad_width), mode='reflect')

        targets.append({
            'node_id': node_idx,
            'name': TOUCHPOINTS[node_idx],
            'dim': dim_required,
            'target': np.round(block, 4)
        })

        current_col = (current_col + dim_required) % DATA_MATRIX.shape[1]

        # Output
        role = "HUB/KNOT" if dim_required >= 16 else ("Core" if dim_required >= 12 else "Linear")
        sig_str = str(block[:3])[:-1] + "...]"
        print(f"{node_idx:<3} | {TOUCHPOINTS[node_idx]:<24} | {role:<12} | {dim_required:<4} | {sig_str}")

    return targets
DSM = np.array([
    [0, 1, 0, 0, 0, 0, 0], # 0
    [0, 0, 1, 1, 0, 0, 0], # 1
    [0, 0, 0, 1, 1, 0, 0], # 2
    [0, 0, 0, 0, 1, 0, 0], # 3
    [0, 0, 0, 0, 0, 1, 0], # 4
    [0, 0, 1, 1, 0, 0, 1], # 5 -> The Feedback Loop (The "Knot")
    [0, 0, 0, 0, 0, 0, 0]  # 6
])
# ======================================================
# 8. EXECUTION
# ======================================================
print("\n--- Notary Service Infrastructure Optimization (7-Node Pinpoint) ---")
print(f"Matrix Shape: {DATA_MATRIX.shape} (Samples x Features)")
print(f"Total Features Registered: {len(FEATURE_KEYS)}")
print("-" * 85)

synthetic_targets = generate_notary_targets(DATA_MATRIX, candidate_dims)

print("-" * 85)
print("Analysis: Graph reduced to 7 critical nodes. 'Identity_Credentialing' and")
print("'Compliance_Audit' remain the high-dimensional Hubs driving the system.")


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx

# Reset random seed for reproducibility
np.random.seed(2025)

# =====================================================
# 1. UNIFIED TOUCHPOINTS (DESIGN + INSURANCE + RISK)
# =====================================================
TOUCHPOINTS = [
    "Clinical_Strategy",     # 0: Bed counts, ROI, Service lines
    "Risk_Underwriting",     # 1: Liability profile, Premium forecasting
    "Site_Catastrophe_Risk", # 2: Flood plains, Seismic, Seismic Insurance
    "Medical_Planning",      # 3: Patient flow, Fall risk, Med-Mal exposure
    "Equipment_Assets",      # 4: MRI/CT (High-value equipment floater)
    "Structural_Integrity",  # 5: Seismic resilience vs. Deductibles
    "MEP_Systems",           # 6: Life safety, HVAC (HEPA), Power backup
    "ICT_Cyber_Security",    # 7: EMR Security, Ransomware insurance
    "Infection_Control",     # 8: Biohazard risk, Liability mitigation
    "Claims_History_Proxy",  # 9: Simulated historical risk data
    "Cost_Engineering",      # 10: CAPEX vs. OPEX (Premium costs)
    "Regulatory_Compliance", # 11: JCI/FGI, License to operate
    "BIM_Digital_Twin",      # 12: Precision underwriting data
    "Reinsurance_Layer",     # 13: Risk transfer for 100M+ assets
    "Operational_Policy"     # 14: Final Policy Issuance / Approval
]

D_GRAPH = len(TOUCHPOINTS)

# =====================================================
# 2. THE DSM (DEPENDENCY STRUCTURE MATRIX)
# =====================================================
# This matrix represents the "Gordian Knot" where Insurance (Risk)
# and Design (Asset) are deeply coupled.
DSM = np.zeros((D_GRAPH, D_GRAPH))

# Key Couplings:
# Risk Underwriting (1) affects Strategy (0) and Cost (10)
DSM[0, 1] = 0.8; DSM[1, 0] = 0.5
# Med Planning (3) affects Infection Control (8) and Claims (9)
DSM[3, 8] = 0.9; DSM[3, 9] = 0.7
# Equipment (4) affects Structural (5) and Reinsurance (13)
DSM[4, 5] = 1.0; DSM[4, 13] = 0.8
# ICT Cyber (7) has a loop with Operational Policy (14)
DSM[7, 14] = 0.6; DSM[14, 7] = 0.9
# Regulatory (11) must approve almost everything
DSM[11, [0, 3, 6, 8]] = 1.0

# =====================================================
# 3. METRIC CONFIGURATION (KPIs)
# =====================================================
METRIC_KEYS = [
    "Loss_Ratio_Prevention", # Insurance perspective
    "Clinical_Safety",       # Hospital perspective
    "Capital_Efficiency",    # Financial perspective
    "Regulatory_Alpha",      # Compliance perspective
]

# Mapping: Node -> [Primary Feature, Impacted Metric]
METRIC_MAP = {
    "Clinical_Strategy":     ["target_roi", "Capital_Efficiency"],
    "Risk_Underwriting":     ["premium_density", "Loss_Ratio_Prevention"],
    "Site_Catastrophe_Risk": ["pml_score", "Loss_Ratio_Prevention"], # Probable Max Loss
    "Medical_Planning":      ["nurse_travel_dist", "Clinical_Safety"],
    "Equipment_Assets":      ["asset_replacement_val", "Capital_Efficiency"],
    "Structural_Integrity":  ["seismic_drift_ratio", "Clinical_Safety"],
    "MEP_Systems":           ["redundancy_n_plus_1", "Clinical_Safety"],
    "ICT_Cyber_Security":    ["encryption_level", "Loss_Ratio_Prevention"],
    "Infection_Control":     ["isolation_room_ratio", "Clinical_Safety"],
    "Claims_History_Proxy":  ["incident_frequency", "Loss_Ratio_Prevention"],
    "Cost_Engineering":      ["total_insured_value", "Capital_Efficiency"],
    "Regulatory_Compliance": ["compliance_score", "Regulatory_Alpha"],
    "BIM_Digital_Twin":      ["data_fidelity_lod", "Regulatory_Alpha"],
    "Reinsurance_Layer":     ["risk_retention_level", "Capital_Efficiency"],
    "Operational_Policy":    ["policy_limit_million", "Regulatory_Alpha"]
}

# =====================================================
# 4. UNIFIED DATA GENERATORS
# =====================================================
def get_generators():
    return {
        "Clinical_Strategy":     lambda: {"target_roi": np.random.uniform(0.08, 0.18)},
        "Risk_Underwriting":     lambda: {"premium_density": np.random.uniform(0.02, 0.05)},
        "Site_Catastrophe_Risk": lambda: {"pml_score": np.random.beta(2, 5)},
        "Medical_Planning":      lambda: {"nurse_travel_dist": np.random.randint(20, 100)},
        "Equipment_Assets":      lambda: {"asset_replacement_val": np.random.uniform(50, 500)}, # Millions
        "Structural_Integrity":  lambda: {"seismic_drift_ratio": np.random.uniform(0.005, 0.02)},
        "MEP_Systems":           lambda: {"redundancy_n_plus_1": np.random.choice([0, 1, 2])},
        "ICT_Cyber_Security":    ["low", "med", "high"], # Handled in formula
        "Infection_Control":     lambda: {"isolation_room_ratio": np.random.uniform(0.1, 0.3)},
        "Claims_History_Proxy":  lambda: {"incident_frequency": np.random.poisson(2)},
        "Cost_Engineering":      lambda: {"total_insured_value": np.random.randint(200, 1500)},
        "Regulatory_Compliance": lambda: {"compliance_score": np.random.uniform(0.85, 1.0)},
        "BIM_Digital_Twin":      lambda: {"data_fidelity_lod": np.random.choice([300, 350, 400, 500])},
        "Reinsurance_Layer":     lambda: {"risk_retention_level": np.random.uniform(0.1, 0.5)},
        "Operational_Policy":    lambda: {"policy_limit_million": np.random.randint(500, 2000)}
    }

# =====================================================
# 5. COMPLEXITY SCORING & NORMALIZATION FORMULAS
# =====================================================
# Transforms raw engineering/insurance data into 0.0-1.0 scores
FORMULAS = {
    "Clinical_Strategy":     lambda x: x / 0.2,
    "Risk_Underwriting":     lambda x: 1.0 - (x * 10),
    "Site_Catastrophe_Risk": lambda x: 1.0 - x, # Lower PML is better
    "Medical_Planning":      lambda x: max(0, 1.0 - (x/120)),
    "Equipment_Assets":      lambda x: x / 500.0,
    "Structural_Integrity":  lambda x: 1.0 - (x / 0.03),
    "MEP_Systems":           lambda x: x / 2.0,
    "ICT_Cyber_Security":    lambda x: 1.0 if x == "high" else 0.5,
    "Infection_Control":     lambda x: x / 0.4,
    "Claims_History_Proxy":  lambda x: max(0, 1.0 - (x/10)),
    "Cost_Engineering":      lambda x: x / 2000.0,
    "Regulatory_Compliance": lambda x: (x - 0.8) / 0.2,
    "BIM_Digital_Twin":      lambda x: x / 500.0,
    "Reinsurance_Layer":     lambda x: 1.0 - x,
    "Operational_Policy":    lambda x: x / 2000.0
}

# =====================================================
# 6. STRUCTURAL SUMMARY
# =====================================================
print("=== UNIFIED INSURANCE-HOSPITAL DESIGN TEMPLATE ===")
print(f"Total Nodes: {len(TOUCHPOINTS)}")
print(f"Metrics Tracked: {METRIC_KEYS}")

# Identifying the 'Heart' of the Insurance Complexity
print("\n--- CORE INSURANCE LOOPS (Coupled Nodes) ---")
# 1. Financial Loop: Strategy <-> Risk Underwriting
# 2. Technical Loop: Cyber Security <-> Operational Policy
# 3. Asset Loop: Equipment <-> Structural <-> Reinsurance
# ======================================================
# UNIFIED GENERATOR MAP (15 TOUCHPOINTS)
# ======================================================
# This map links each Touchpoint to its specific data generator function.
# It handles both the 12 design nodes and the 3 new insurance/risk nodes.

GENERATOR_MAP = {
    # --- DESIGN & CLINICAL NODES ---
    "Clinical_Strategy":     lambda: {
        "target_roi": np.random.uniform(0.08, 0.18),
        "icu_bed_count": np.random.randint(10, 60)
    },
    "Site_Catastrophe_Risk": lambda: {
        "pml_score": np.random.beta(2, 5), # Probable Max Loss
        "seismic_zone": np.random.choice([1, 2, 3, 4])
    },
    "Medical_Planning":      lambda: {
        "nurse_travel_dist": np.random.randint(20, 100),
        "fall_risk_index": np.random.uniform(0.1, 0.9)
    },
    "Equipment_Assets":      lambda: {
        "asset_replacement_val": np.random.uniform(50, 500), # Millions
        "mri_shielding_req": np.random.uniform(0.5, 1.0)
    },
    "Structural_Integrity":  lambda: {
        "seismic_drift_ratio": np.random.uniform(0.005, 0.02),
        "vibration_sensitivity": np.random.choice(["VC-A", "VC-B", "VC-C"])
    },
    "MEP_Systems":           lambda: {
        "redundancy_n_plus_1": np.random.choice([0, 1, 2]),
        "ach_operating_room": np.random.randint(15, 25)
    },
    "ICT_Cyber_Security":    lambda: {
        "encryption_level": np.random.choice(["low", "med", "high"]),
        "vulnerability_count": np.random.randint(0, 50)
    },
    "Infection_Control":     lambda: {
        "isolation_room_ratio": np.random.uniform(0.1, 0.3),
        "hepa_coverage": np.random.uniform(0.7, 1.0)
    },
    "Claims_History_Proxy":  lambda: {
        "incident_frequency": np.random.poisson(2),
        "projected_annual_claims": np.random.randint(1, 15)
    },
    "Cost_Engineering":      lambda: {
        "total_insured_value": np.random.randint(200, 1500),
        "ve_savings_ratio": np.random.uniform(0.05, 0.20)
    },
    "Regulatory_Compliance": lambda: {
        "compliance_score": np.random.uniform(0.85, 1.0),
        "fgi_compliance_gap": np.random.randint(0, 5)
    },
    "BIM_Digital_Twin":      lambda: {
        "data_fidelity_lod": np.random.choice([300, 350, 400, 500]),
        "clash_resolution_rate": np.random.uniform(0.6, 0.99)
    },

    # --- INSURANCE & GOVERNANCE NODES ---
    "Risk_Underwriting":     lambda: {
        "premium_density": np.random.uniform(0.02, 0.05),
        "liability_exposure_index": np.random.uniform(0.1, 0.8)
    },
    "Reinsurance_Layer":     lambda: {
        "risk_retention_level": np.random.uniform(0.1, 0.5),
        "reinsurance_attachment_point": np.random.randint(10, 50)
    },
    "Operational_Policy":    lambda: {
        "policy_limit_million": np.random.randint(500, 2000),
        "policy_compliance_score": np.random.uniform(0.7, 1.0)
    }
}

import numpy as np

# ======================================================
# 1. METRIC KEYS (15 Touchpoints: Design + Insurance)
# ======================================================
METRIC_KEYS = [
    "ClinicalProgramming",    # 0
    "SiteLogistics",          # 1
    "MedicalEquipPlanning",   # 2
    "ArchitecturalLayout",    # 3
    "InfectionControl",       # 4
    "StructuralSeismic",      # 5
    "SpecializedMEP",         # 6
    "ICT_SmartHospital",      # 7
    "Regulatory_Accred",      # 8
    "ValueEngineering",       # 9
    "BIM_DigitalTwin",        # 10
    "Risk_Underwriting",      # 11: NEW - Liability & Premiums
    "Reinsurance_Layer",      # 12: NEW - Catastrophic Risk Transfer
    "Operational_Policy",     # 13: NEW - Governance & Procedures
    "Aggregate_Safety"        # 14: Summary Index
]

# ======================================================
# 2. FEATURE KEYS (Expanded for Insurance Domain)
# ======================================================
FEATURE_KEYS = [
    # Clinical / Site
    'icu_bed_count', 'acuity_level', 'theatre_complexity',
    'ambulance_access_rating', 'helipad_clearance',
    # Technical
    'mri_shielding_req', 'radiation_protection_mm',
    'clinical_adjacency_score', 'infection_control_zoning',
    'vibration_sensitivity_vc', 'seismic_importance_factor',
    'ach_operating_room', 'medgas_redundancy',
    'telemetry_coverage', 'nurse_call_latency',
    # Compliance / Process
    'fgi_compliance_gap', 'jci_safety_score',
    've_savings_ratio', 'clash_resolution_rate',
    'handover_completeness',
    # Insurance / Financial (NEW)
    'premium_base_rate', 'liability_exposure_index',
    'risk_retention_limit', 'reinsurance_attachment_point',
    'policy_compliance_score'
]

# ======================================================
# 3. MAPPING (Node -> Features)
# ======================================================
metric_feature_map = {
    "ClinicalProgramming":   ['icu_bed_count', 'theatre_complexity'],
    "SiteLogistics":         ['ambulance_access_rating', 'helipad_clearance'],
    "MedicalEquipPlanning":  ['mri_shielding_req', 'radiation_protection_mm'],
    "ArchitecturalLayout":   ['clinical_adjacency_score', 'infection_control_zoning'],
    "InfectionControl":      ['ach_operating_room', 'infection_control_zoning'],
    "StructuralSeismic":     ['vibration_sensitivity_vc', 'seismic_importance_factor'],
    "SpecializedMEP":        ['medgas_redundancy', 'ach_operating_room'],
    "ICT_SmartHospital":     ['telemetry_coverage', 'nurse_call_latency'],
    "Regulatory_Accred":     ['fgi_compliance_gap', 'jci_safety_score'],
    "ValueEngineering":      ['ve_savings_ratio', 'theatre_complexity'],
    "BIM_DigitalTwin":       ['clash_resolution_rate', 'handover_completeness'],
    "Risk_Underwriting":     ['premium_base_rate', 'liability_exposure_index'],
    "Reinsurance_Layer":     ['risk_retention_limit', 'reinsurance_attachment_point'],
    "Operational_Policy":    ['policy_compliance_score', 'jci_safety_score'],
    "Aggregate_Safety":      ['jci_safety_score', 'clash_resolution_rate', 'liability_exposure_index']
}

# ======================================================
# 4. FORMULAS (Physics, Economics & Actuarial)
# ======================================================
METRIC_FORMULAS = [
    lambda x: np.log(x + 10),            # 0. Clinical
    lambda x: np.exp(-x),               # 1. Site
    lambda x: x**1.8 / 10.0,            # 2. MedEquip
    lambda x: x * 1.2,                  # 3. Arch
    lambda x: 3.0 * x,                  # 4. Infection
    lambda x: 2.5 * x,                  # 5. Structural
    lambda x: np.tanh(x) * 2.0,         # 6. MEP
    lambda x: 1.0 / (1.0 + x),          # 7. ICT
    lambda x: np.exp(-x * 3.0),         # 8. Regulatory
    lambda x: 1.0 / (x + 0.1),          # 9. ValueEng
    lambda x: (np.arctan(x) * 2)/np.pi, # 10. BIM
    lambda x: 1.0 / (x * 0.5 + 0.1),    # 11. Underwriting: Lower exposure = Higher Score
    lambda x: x * 0.8,                  # 12. Reinsurance: Higher retention capacity is good
    lambda x: np.log(x + 2),            # 13. Policy: Log benefit of strict governance
    lambda x: np.mean(x),               # 14. Aggregate
]

# ======================================================
# 5. TARGET MATRIX (15x15 Interaction Graph)
# ======================================================
# Row = Source Node, Column = Target Metric Impact
# 0-14 Indices
METRIC_TARGET = [
    # 0  1  2  3  4  5  6  7  8  9  10 11 12 13 14
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1], # 0: Clinical -> VE, Underwriting
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1], # 1: Site -> VE, Underwriting
    [0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1], # 2: MedEquip -> Reinsurance (High Asset Value)
    [0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1], # 3: Arch -> Policy (Flows)
    [0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1], # 4: Infection -> Underwriting, Policy
    [0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1], # 5: Structural -> Reinsurance
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1], # 6: MEP -> Policy
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1], # 7: ICT -> Underwriting (Cyber), Policy
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1], # 8: Regulatory (Global Constraint)
    [1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1], # 9: Value Eng (Feedback Loop)
    [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], # 10: BIM (Global Coordination)
    [1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1], # 11: Underwriting -> Feedbacks to Strategy/Reins
    [0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1], # 12: Reinsurance -> Feedbacks to MedEquip/Struct
    [0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1], # 13: Policy -> Feedbacks to Operations
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], # 14: Aggregate
]

# ======================================================
# 6. OPTIMIZER (Branch & Bound)
# ======================================================
class BranchBoundOptimizer:
    def __init__(self, tol=1e-3, max_depth=20, minimize=True, value_range=(0.0, 10.0)):
        self.tol = tol
        self.max_depth = max_depth
        self.minimize = minimize
        self.value_range = value_range

    def optimize(self, features, y=None, metric_mask=None):
        y = np.zeros(3) if y is None else np.array(y[:3])
        base = np.mean(list(features.values())) + np.mean(y)

        a0, b0 = self.value_range
        a0 += base
        b0 += base

        work = [(a0, b0, 0)]
        best_x = None
        best_score = np.inf if self.minimize else -np.inf

        def better(s1, s2):
            return s1 < s2 if self.minimize else s1 > s2

        while work:
            a, b, depth = work.pop()
            mid = 0.5 * (a + b)

            # Apply Formula if mask is active
            mv = [f(mid) if m else 0.0 for f, m in zip(METRIC_FORMULAS, metric_mask)]
            score = sum(mv)

            if best_x is None or better(score, best_score):
                best_x = mid
                best_score = score

            if depth >= self.max_depth or (b - a) < self.tol:
                continue

            work.append((a, mid, depth + 1))
            work.append((mid, b, depth + 1))

        return best_x

# ======================================================
# 8. EVALUATOR
# ======================================================
class MetricsEvaluator:
    def __init__(
        self,
        data_matrix,
        metric_formulas=METRIC_FORMULAS,
        metric_feature_map=metric_feature_map,
        feature_keys=FEATURE_KEYS,
        feature_target=None,
        metric_target=None,
        tol=1e-3,
        max_depth=20,
        minimize=False,
        value_range=(0.0, 5.0)
    ):
        self.data_matrix = data_matrix
        self.metric_formulas = metric_formulas
        self.metric_feature_map = metric_feature_map
        self.feature_keys = feature_keys

        # Defaults
        self.feature_target = feature_target or [[1]*len(feature_keys) for _ in range(data_matrix.shape[0])]
        self.metric_target = metric_target or METRIC_TARGET
        self.num_nodes = data_matrix.shape[0]

        self.optimizer = BranchBoundOptimizer(
            tol=tol,
            max_depth=max_depth,
            minimize=minimize,
            value_range=value_range
        )

    def extract_features(self, node_idx):
        if node_idx >= len(self.data_matrix): return {}
        row = self.data_matrix[node_idx]
        mask = self.feature_target[node_idx]
        features = {k: v for k, v, m in zip(self.feature_keys, row, mask) if m}
        return features

    def compute_node_metrics(self, node_idx, y=None):
        features = self.extract_features(node_idx)
        metric_mask = self.metric_target[node_idx] if node_idx < len(self.metric_target) else [0]*len(METRIC_KEYS)
        metric_values = {}

        for key, formula, mask in zip(METRIC_KEYS, self.metric_formulas, metric_mask):
            if mask and key in self.metric_feature_map:
                relevant_keys = self.metric_feature_map[key]
                relevant_features = [features[f] for f in relevant_keys if f in features]
                x = np.mean(relevant_features) if relevant_features else 0.0

                # Optimize
                opt_value = self.optimizer.optimize(features={key: x}, y=y, metric_mask=[1])
                metric_values[key] = formula(opt_value)
            else:
                metric_values[key] = 0.0

        metric_values['score'] = sum(metric_values.values())
        return metric_values
# ======================================================
# 8. EXECUTION
# ======================================================

# Generate Mock Data (15 Nodes x 25 Features)
# ======================================================
# 9. DIMENSIONALITY CONFIGURATION (15 Nodes)
# ======================================================

# Dimensions assigned based on node complexity and coupling depth.
# "Hub" nodes and Financial nodes get higher dimensions.

candidate_dims = [
    [4],   # 0: ClinicalProgramming (Input)
    [4],   # 1: SiteLogistics (Input)
    [16],  # 2: MedicalEquipPlanning (HUB: High Value Assets)
    [12],  # 3: ArchitecturalLayout (Coupled)
    [12],  # 4: InfectionControl (Coupled)
    [12],  # 5: StructuralSeismic (Coupled)
    [16],  # 6: SpecializedMEP (HUB: The Engine)
    [8],   # 7: ICT_SmartHospital (High Tech)
    [12],  # 8: Regulatory_Accred (Constraint)
    [8],   # 9: ValueEngineering (Feedback)
    [12],  # 10: BIM_DigitalTwin (Coordination Layer)
    [16],  # 11: Risk_Underwriting (NEW: High Complexity Financial)
    [16],  # 12: Reinsurance_Layer (NEW: High Value/Complex)
    [8],   # 13: Operational_Policy (Process)
    [4]    # 14: Aggregate_Safety (Output)
]

D_graph = len(candidate_dims)
# 1. Ensure TOUCHPOINTS match the GENERATOR_MAP keys exactly
TOUCHPOINTS = [
    "Clinical_Strategy",     # Matches Generator Map
    "Risk_Underwriting",
    "Site_Catastrophe_Risk",
    "Medical_Planning",
    "Equipment_Assets",
    "Structural_Integrity",
    "MEP_Systems",
    "ICT_Cyber_Security",
    "Infection_Control",
    "Claims_History_Proxy",
    "Cost_Engineering",
    "Regulatory_Compliance",
    "BIM_Digital_Twin",
    "Reinsurance_Layer",
    "Operational_Policy"
]

# 2. Re-run the Data Matrix generation
num_samples = 100
feature_list = []

for tp in TOUCHPOINTS:
    # Now tp ("Clinical_Strategy") will correctly find the key in GENERATOR_MAP
    samples = [GENERATOR_MAP[tp]() for _ in range(num_samples)]
    df = pd.DataFrame(samples)

    # Encode categorical text (like 'high'/'med'/'low') into numbers
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.factorize(df[col])[0]

    feature_list.append(df)

# 3. Build and Normalize
DATA_MATRIX_RAW = pd.concat(feature_list, axis=1).to_numpy()
DATA_MATRIX = (DATA_MATRIX_RAW - DATA_MATRIX_RAW.min(axis=0)) / (np.ptp(DATA_MATRIX_RAW, axis=0) + 1e-8)

print(f"Success! Data Matrix Shape: {DATA_MATRIX.shape}")
# =====================================================
# 3. HIGH-FIDELITY SYNTHETIC TARGET GENERATOR
# =====================================================
def generate_synthetic_targets_per_node(DATA_MATRIX, candidate_dims):
    """
    Transforms the normalized feature space into node-specific target signatures.
    Uses reflective padding to preserve variance in high-dimensional risk nodes.
    """
    dims_flat = [i[0] for i in candidate_dims]
    num_nodes = len(dims_flat)
    targets = []
    current_col = 0

    for node_idx in range(num_nodes):
        dim = dims_flat[node_idx]

        # Select the feature row for this specific touchpoint
        # Wrapping ensures we don't index out of bounds if num_nodes > num_samples
        row = DATA_MATRIX[node_idx % DATA_MATRIX.shape[0]]

        # Extract a contiguous block of data corresponding to the node's dimensionality
        block = row[current_col : current_col + dim]

        # Padding logic: Crucial for nodes requiring higher dimensions (16)
        # than their direct feature count. Reflective padding maintains the
        # 'extreme' values essential for catastrophic risk modeling.
        if len(block) < dim:
            block = np.pad(block, (0, dim - len(block)), mode='reflect')

        targets.append({
            'node_id': node_idx,
            'touchpoint': TOUCHPOINTS[node_idx],
            'dim_required': dim,
            'target': np.round(block, 4)
        })

        # Increment the column pointer with a modular wrap to recycle feature variance
        current_col = (current_col + dim) % DATA_MATRIX.shape[1]

    return targets

# Execute target generation
synthetic_targets = generate_synthetic_targets_per_node(DATA_MATRIX, candidate_dims)

# =====================================================
# 4. DIAGNOSTIC OUTPUT
# =====================================================
print(f"--- 15-Node Optimization Matrix Created ---")
print(f"Total Combined Features: {DATA_MATRIX.shape[1]}")
print("-" * 50)

for t in synthetic_targets:
    node_desc = f"{t['node_id']:02d} | {t['touchpoint']:<22}"
    print(f"Node {node_desc} | Size: {t['dim_required']:02d} | Sig: {t['target'][:4]}...")

# Service TP optimization and TP ascociation optimzation

In [ ]:

# ======================================================
# 6. OPTIMIZER (Branch & Bound)
# ======================================================
class BranchBoundOptimizer:
    def __init__(self, tol=1e-3, max_depth=20, minimize=True, value_range=(0.0, 10.0)):
        self.tol = tol
        self.max_depth = max_depth
        self.minimize = minimize
        self.value_range = value_range

    def optimize(self, features, y=None, metric_mask=None):
        y = np.zeros(3) if y is None else np.array(y[:3])
        base = np.mean(list(features.values())) + np.mean(y)

        a0, b0 = self.value_range
        a0 += base
        b0 += base

        work = [(a0, b0, 0)]
        best_x = None
        best_score = np.inf if self.minimize else -np.inf

        def better(s1, s2):
            return s1 < s2 if self.minimize else s1 > s2

        while work:
            a, b, depth = work.pop()
            mid = 0.5 * (a + b)

            # Apply Formula if mask is active
            mv = [f(mid) if m else 0.0 for f, m in zip(METRIC_FORMULAS, metric_mask)]
            score = sum(mv)

            if best_x is None or better(score, best_score):
                best_x = mid
                best_score = score

            if depth >= self.max_depth or (b - a) < self.tol:
                continue

            work.append((a, mid, depth + 1))
            work.append((mid, b, depth + 1))

        return best_x

# ======================================================
# 8. EVALUATOR
# ======================================================
class MetricsEvaluator:
    def __init__(
        self,
        data_matrix,
        metric_formulas=METRIC_FORMULAS,
        metric_feature_map=metric_feature_map,
        feature_keys=FEATURE_KEYS,
        feature_target=None,
        metric_target=None,
        tol=1e-3,
        max_depth=20,
        minimize=False,
        value_range=(0.0, 5.0)
    ):
        self.data_matrix = data_matrix
        self.metric_formulas = metric_formulas
        self.metric_feature_map = metric_feature_map
        self.feature_keys = feature_keys

        # Defaults
        self.feature_target = feature_target or [[1]*len(feature_keys) for _ in range(data_matrix.shape[0])]
        self.metric_target = metric_target or METRIC_TARGET
        self.num_nodes = data_matrix.shape[0]

        self.optimizer = BranchBoundOptimizer(
            tol=tol,
            max_depth=max_depth,
            minimize=minimize,
            value_range=value_range
        )

    def extract_features(self, node_idx):
        if node_idx >= len(self.data_matrix): return {}
        row = self.data_matrix[node_idx]
        mask = self.feature_target[node_idx]
        features = {k: v for k, v, m in zip(self.feature_keys, row, mask) if m}
        return features

    def compute_node_metrics(self, node_idx, y=None):
        features = self.extract_features(node_idx)
        metric_mask = self.metric_target[node_idx] if node_idx < len(self.metric_target) else [0]*len(METRIC_KEYS)
        metric_values = {}

        for key, formula, mask in zip(METRIC_KEYS, self.metric_formulas, metric_mask):
            if mask and key in self.metric_feature_map:
                relevant_keys = self.metric_feature_map[key]
                relevant_features = [features[f] for f in relevant_keys if f in features]
                x = np.mean(relevant_features) if relevant_features else 0.0

                # Optimize
                opt_value = self.optimizer.optimize(features={key: x}, y=y, metric_mask=[1])
                metric_values[key] = formula(opt_value)
            else:
                metric_values[key] = 0.0

        metric_values['score'] = sum(metric_values.values())
        return metric_values
# ======================================================
# 8. EXECUTION
# ======================================================

# Generate Mock Data (15 Nodes x 25 Features)
# ======================================================
# 9. DIMENSIONALITY CONFIGURATION (15 Nodes)
# ======================================================

# Dimensions assigned based on node complexity and coupling depth.
# "Hub" nodes and Financial nodes get higher dimensions.
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from typing import List

# ======================================================
# 1. THE GRAPH NETWORK (MODEL ARCHITECTURE)
# ======================================================
class GraphNetwork(nn.Module):
    def __init__(self,
                 node_feature_dims: List[int],
                 adjacency_matrix: np.ndarray,
                 latent_dim: int = 16):
        super().__init__()

        self.num_nodes = len(node_feature_dims)
        self.node_feature_dims = node_feature_dims
        self.latent_dim = latent_dim

        # Register the Graph Structure (DSM)
        self.register_buffer('adj', torch.tensor(adjacency_matrix, dtype=torch.float32).t())

        # 1. Dynamic Node Encoders
        self.encoders = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, 16),
                nn.Tanh(), # Tanh is safer than ELU for preventing early NaN divergence
                nn.Linear(16, latent_dim),
                nn.LayerNorm(latent_dim)
            ) for dim in node_feature_dims
        ])

        # 2. Message Passing Layer
        self.message_passing = nn.Linear(latent_dim, latent_dim)

        # 3. Metric Projection Heads
        self.metric_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(latent_dim, 8),
                nn.ReLU(),
                nn.Linear(8, 1),
                nn.Sigmoid() # Forces output to 0.0 - 1.0
            ) for _ in range(self.num_nodes)
        ])

        # Initialize weights safely
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x_full):
        batch_size = x_full.size(0)
        node_embeddings = []
        current_col = 0

        # --- Step 1: Feature Slicing & Encoding ---
        for i, encoder in enumerate(self.encoders):
            dim_expected = self.node_feature_dims[i]

            # Slice data for this specific node
            x_slice = x_full[:, current_col : current_col + dim_expected]

            # Padding logic if input is too small
            if x_slice.shape[1] < dim_expected:
                padding = torch.zeros((batch_size, dim_expected - x_slice.shape[1]), device=x_full.device)
                x_slice = torch.cat([x_slice, padding], dim=1)

            h = encoder(x_slice)
            node_embeddings.append(h)
            current_col += dim_expected

        # --- Step 2: Graph Diffusion ---
        H = torch.stack(node_embeddings, dim=1)
        H_transformed = self.message_passing(H)

        # Einstein Summation: Mix latent states based on Adjacency Matrix
        H_propagated = torch.einsum('ij, bjk -> bik', self.adj, H_transformed)

        # Residual + Tanh Activation
        H_fused = torch.tanh(H + H_propagated)

        # --- Step 3: Metric Derivation ---
        metrics = [self.metric_heads[i](H_fused[:, i, :]) for i in range(self.num_nodes)]

        return torch.cat(metrics, dim=1), H_fused

# ======================================================
# 2. DATA SETUP (RUNNABLE EXAMPLE)
# ======================================================
def setup_synthetic_data():
    print("--- Generatng Synthetic Data ---")

    # Configuration matches your previous description (7 Nodes)
    input_dims_flat = [d[0] for d in candidate_dims]
    num_nodes = len(input_dims_flat)
    total_features = sum(input_dims_flat)
    num_samples = 500

    # 1. Create Raw Features (X)
    X_raw = np.random.randn(num_samples, total_features).astype(np.float32)
    # Sanitize inputs (remove extreme outliers)
    X_raw = np.clip(X_raw, -3.0, 3.0)
    X_tensor = torch.FloatTensor(X_raw)

    # 2. Create Targets (Y) - GROUND TRUTH
    # We create fake "correct" answers strictly between 0 and 1
    Y_list = []
    current_col = 0
    for dim in input_dims_flat:
        # Fake logic: Average of features, normalized to 0-1
        feats = X_raw[:, current_col : current_col + dim]

        # Simple non-linear formula to give the network something to learn
        # e.g., Sigmoid(Sum(features))
        val = 1.0 / (1.0 + np.exp(-np.sum(feats, axis=1)))

        Y_list.append(val)
        current_col += dim

    Y_tensor = torch.FloatTensor(np.stack(Y_list, axis=1))

    # 3. Create Topology (DSM)
    # Simple chain + one feedback loop
    DSM = np.eye(num_nodes) # Self loops
    for i in range(num_nodes - 1):
        DSM[i, i+1] = 1 # Chain
    DSM[num_nodes-1, 0] = 1 # Feedback loop (Last -> First)

    return X_tensor, Y_tensor, DSM, input_dims_flat

# ======================================================
# 3. TRAINING LOOP
# ======================================================
def train_graph_network():
    # 1. Get Data
    X_train, Y_train, DSM, dims = setup_synthetic_data()

    # 2. Initialize Model
    model = GraphNetwork(
        node_feature_dims=dims,
        adjacency_matrix=DSM,
        latent_dim=32
    )

    # 3. Training Config
    optimizer = optim.Adam(model.parameters(), lr=0.002)
    criterion = nn.MSELoss() # Regression loss
    epochs = 300
    loss_history = []

    print(f"\n--- Starting Training on {len(X_train)} samples ---")
    model.train() # Enable training mode (gradients on)

    for epoch in range(epochs):
        optimizer.zero_grad() # Reset gradients

        # Forward Pass
        # pred_metrics comes out of Sigmoid, so it's 0.0 to 1.0
        pred_metrics, _ = model(X_train)

        # Calculate Loss
        loss = criterion(pred_metrics, Y_train)

        # Backward Pass
        loss.backward()

        # --- CRITICAL: GRADIENT CLIPPING ---
        # Prevents "Loss: nan" by capping the gradient size
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step() # Update weights

        # Track history
        loss_history.append(loss.item())

        if (epoch + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.6f}")

    # 4. Visualization
    plt.figure(figsize=(10, 5))
    plt.plot(loss_history, label='Training Loss', color='teal')
    plt.title('GraphNetwork Learning Curve')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print("Training Complete.")
    return model

# ======================================================
# EXECUTION
# ======================================================
if __name__ == "__main__":
    trained_model = train_graph_network()

# Fuzzy Hierarchical Plex-Multiplex

---



In [ ]:


# =============================================================================
# EXTERNAL DEPENDENCIES & CONFIGURATION
# =============================================================================
# These variables are referenced in the original code but not defined.
# Assumed to be present in the execution environment.
# -----------------------------------------------------------------------------
# D_graph = ...
# DATA_MATRIX = ...
# METRIC_KEYS = [...]
# METRIC_TARGET = [...]
# METRIC_FORMULAS = [...]
# METRIC_INVERSES = {...}
# synthetic_targets = ...
# MetricsEvaluator = ... (Class)
# -----------------------------------------------------------------------------
# =============================================================================
# NEW: DETERMINISTIC DFS & PATH EVALUATOR
# =============================================================================


# =============================================================================
# HELPER CLASSES
# =============================================================================
class DSM_Tracker:
    """
    Tracks a DSM (Design Structure Matrix) layer and its residual.
    """
    def __init__(self, multiplex_layer):
        self.layer = multiplex_layer
        self.primary_dsm = None
        self.residual_dsm = None
        self.update_dsms()

    def update_dsms(self):
        if self.primary_dsm is None:
            # First time: store current DSM as reference
            self.primary_dsm = self.layer.chosen_Gmat.copy()
            self.residual_dsm = np.zeros_like(self.primary_dsm)
        else:
            # Update residual: current DSM minus primary
            current = self.layer.chosen_Gmat
            self.residual_dsm = current - self.primary_dsm

    def get_matrices(self):
        return self.primary_dsm, self.residual_dsm

    def print_matrices(self):
        print("\n--- Primary DSM ---")
        print(self.primary_dsm)
        print("\n--- Residual DSM ---")
        print(self.residual_dsm)


import numpy as np

class DSM_Layer_Decomposer:
    """
    Manages the additive decomposition of DSM layers.
    Now includes integrity checks and baseline-aware reconstruction.
    """
    def __init__(self, baseline_matrix, mode='additive'):
        self.baseline_matrix = baseline_matrix.copy()
        self.current_total = baseline_matrix.copy()
        self.mode = mode
        self.layers = []
        self.residuals = []

    def add_snapshot(self, new_total_matrix):
        """
        Calculates the DELTA (change) between the new state and the previous state.
        """
        # Calculate Delta
        delta = new_total_matrix - self.current_total

        # Guard clause: Optional warning if delta is empty
        if np.all(delta == 0):
            print(f"Warning: Snapshot added with ZERO change. Layer {len(self.layers)} is empty.")

        self.layers.append(delta.copy())

        # Update current tracker
        self.current_total = new_total_matrix.copy()

        # Calculate residual (Total Drift from Baseline)
        residual = self.current_total - self.baseline_matrix
        self.residuals.append(residual)

        layer_id = len(self.layers) - 1
        print(f"\n=== DSM LAYER {layer_id} CAPTURED ===")
        # formatting for cleaner output
        print(f"Layer Contribution (Delta) Max Value: {np.max(np.abs(delta)):.3f}")

        return delta

    def get_reconstruction(self, include_baseline=True):
        """
        reconstructs the matrix.
        Args:
            include_baseline (bool): If True, returns (Baseline + Layers).
                                     If False, returns only the changes (Layers).
        """
        layers_sum = np.sum(self.layers, axis=0)

        if include_baseline:
            return self.baseline_matrix + layers_sum
        return layers_sum

    def verify_integrity(self):
        """Debug function to prove all layers are accounted for."""
        reconstructed = self.get_reconstruction(include_baseline=True)
        is_correct = np.allclose(reconstructed, self.current_total)
        print(f"Integrity Check: {'PASSED' if is_correct else 'FAILED'}")
        print(f"Total Layers Stored: {len(self.layers)}")

def build_dsm_from_walks(D, paths):
    """
    Constructs a DSM where entry [i,j] is the probability
    that a successful process moves from i to j.
    """
    flow = np.zeros((D, D))

    # Count transitions
    for path in paths:
        for k in range(len(path) - 1):
            u, v = path[k], path[k+1]
            flow[u, v] += 1

    # Normalize by the total number of successful walks.
    # This prevents 'saturation'—if an edge is rarely used, it stays small.
    n_paths = len(paths)
    if n_paths > 0:
        flow = flow / n_paths

    np.fill_diagonal(flow, 0.0)
    return flow



try:
    D_GRAPH = D_graph
except:
    D_graph=D_GRAPH


def build_coupling_weight(label, m_i, m_j):
    """
    Interpretation: Pascal's Law P = F / A
    Force (F) = Potential delta in metrics (Improvement)
    Area (A) = Complexity/Friction of the destination node
    Valve = Orifice efficiency based on coupling type
    """

    # 1. THE NODAL VECTORS
    v_i = np.array(list(m_i.values())) if isinstance(m_i, dict) else np.array(m_i)
    v_j = np.array(list(m_j.values())) if isinstance(m_j, dict) else np.array(m_j)

    # 2. THE DRIVING FORCE (F)
    # The pressure only builds if the destination offers more "potential"
    # F = || max(0, v_j - v_i) ||
    force = np.linalg.norm(np.maximum(v_j - v_i, 0)) + 1e-9

    # 3. THE SURFACE AREA (A)
    # The 'wider' the destination node (more complex/costly),
    # the more the force is distributed, lowering the pressure.
    surface_area = np.mean(v_j) + 0.1

    # 4. THE VALVE ORIFICE (Efficiency)
    # A-type couplings are 'Wide Nozzles' (High pressure translation)
    # B-type couplings are 'Constricted Nozzles' (Damped flow)
    label_str = str(label) if label is not None else ""
    if "A" in label_str:
        motivator = force # High flow efficiency
    elif "B" in label_str:
        motivator =  1.0  # Damped efficiency
    else:
        motivator = surface_area# Standard atmospheric pressure

    # 5. THE SYSTEM PRESSURE (P)
    # (i) Power driver
    # (iii) Force driver
    # (ii) Pressure
    pressure = motivator * (force / surface_area)

    return max(pressure, 0.0001)
import numpy as np

def infer_node_types(node_metrics, candidate_dims=candidate_dims):
    dims_flat = [d[0] for d in candidate_dims]

    min_dim = min(dims_flat)
    max_dim = max(dims_flat)
    median_dim = np.median(dims_flat)

    node_types = []
    for d in dims_flat:
        if d == min_dim:
            node_types.append("A") # Edge (Output/Market)
        elif d == max_dim:
            node_types.append("A") # Core (16D Hubs/Insurance)
        elif d < median_dim:
            node_types.append("A") # External Gatekeeper (New!)
        else:
            node_types.append("A") # Internal Hub (Cultivation)

    return node_types

def build_coupling_matrix(node_types):
    """
    Generic Behavioral Matrix Builder.
    Uses a 'Transition Rulebook' to enforce flow constraints.
    """
    D = len(node_types)
    C = np.full((D, D), "VOID", dtype=object)

    # DEFINING THE FLOW LOGIC (The "Constitution")
    # Format: { Source_Type: { Allowed_Dest_Type: Label } }
    # Any transition not listed here becomes "ABSORBED"



    RULES = {
    "D": {
        "D": "D->D",      # Lateral Saturation (Peer-Hopping Allowed)
        "C": "D->C",      # Saturated Data feeds Core
        "B": "D->B"       # External triggers Hub directly
    },
    "C": {
        "B": "C->B"       # Core feeds Hub
    },
    "B": {
        "B": "B->B",      # Hub stability
        "A": "B->A"       # Hub ejects to Edge
    },
    "A": {
        "A": "A->A",      # Edge local routing
        "C": "A->C",      # Edge returns to Core
        "D": "A->D"       # Edge triggers external refresh
    }
}
    D_size = len(node_types)
    C = np.full((D_size, D_size), "VOID", dtype=object)

    for i in range(D_size):
        src = node_types[i]
        for j in range(D_size):
            dst = node_types[j]

            if src in RULES:
                allowed_moves = RULES[src]

                if dst in allowed_moves:
                    label = allowed_moves[dst]

                    # BLOCK Hub-Hopping for B (Efficiency constraint)
                    if src == "B" and dst == "B" and i != j:
                        C[i, j] = "ABSORBED"

                    # ALLOW Peer-Hopping for D (Saturation constraint)
                    # This allows different D-nodes to cross-talk
                    elif src == "D" and dst == "D":
                        C[i, j] = "D<->D" if i != j else "D_SELF"

                    else:
                        C[i, j] = label
                else:
                    C[i, j] = "ABSORBED"
            else:
                C[i, j] = "ABSORBED"

    return C

def build_dsm_from_walks(D, paths):
    """
    Constructs a DSM (Design Structure Matrix) from walk paths.
    Entry [i,j] is the probability that a process moves from i to j.
    """
    flow = np.zeros((D, D))
    for path in paths:
        for k in range(len(path) - 1):
            u, v = path[k], path[k+1]
            flow[u, v] += 1

    n_paths = len(paths)
    if n_paths > 0:
        flow = flow / n_paths

    np.fill_diagonal(flow, 0.0)
    return flow

# =============================================================================
# 2. THE METRIC-DRIVEN RANDOM WALKER (Consolidated)
# =============================================================================


In [ ]:

import math
import numpy as np
import math


def cosine_similarity(a, b, p = '0'):
    if not p :
        """
        Compute true cosine similarity between two vectors a and b.
        Returns a value in [-1, 1].
        """
        dot_product = sum(x * y for x, y in zip(a, b))
        norm_a = math.sqrt(sum(x ** 2 for x in a))
        norm_b = math.sqrt(sum(y ** 2 for y in b))

        if norm_a == 0 or norm_b == 0:
            return 0.0  # define similarity as 0 if any vector is zero

        return dot_product / (norm_a * norm_b)
    elif p=='A' :
        dot_product = np.dot(a, b)
        norm_a = np.linalg.norm(a)
        norm_b = np.linalg.norm(b)

        if norm_a == 0 or norm_b == 0:
            return 1.0 # Maximum distance if a vector is empty

        cos_sim = dot_product / (norm_a * norm_b)

        # 2. Clip to handle floating point errors outside [-1, 1]
        cos_sim = max(min(cos_sim, 1.0), -1.0)

        # 3. Convert to Angular Distance
        # This linearizes the result, making it robust in 25D space
        return math.acos(cos_sim) / math.pi
    elif p == '0':
        """
        Computes Hermitian Cosine Similarity using Complex Numbers.
        Encodes local structure (phase) for higher sensitivity.
        """
        # 1. Convert to Float Arrays
        va = np.array(a, dtype=np.float64)
        vb = np.array(b, dtype=np.float64)

        # 2. Project into Complex Plane: Z = Real + i*Imaginary
        # We create the Imaginary part by 'rolling' the data (shifting by 1).
        # This encodes the relationship between neighbor values.
        za = va + 1j * np.roll(va, 1)
        zb = vb + 1j * np.roll(vb, 1)

        # 3. Hermitian Dot Product: dot = sum(a * conj(b))
        # np.vdot handles complex conjugation automatically
        dot_product = np.vdot(za, zb)

        # 4. Compute Complex Magnitudes (Norms)
        norm_a = np.linalg.norm(za)
        norm_b = np.linalg.norm(zb)

        if norm_a == 0 or norm_b == 0:
            return 0.0

        # 5. Normalize and return Magnitude
        # We take np.abs() because the result is a complex number.
        similarity = np.abs(dot_product) / (norm_a * norm_b)

        return float(similarity)
    else:
        if len(a) != len(b):
            raise ValueError("Vectors must have the same length")

        # Sum of absolute d#ifferences raised to the power of p
        sum_diff = sum(abs(x - y) ** p for x, y in zip(a, b))

        # Take the p-th root of the sum
        return sum_diff ** (1.0 / p)


def build_transition_matrix(data, state_matrix, min_similarity=0.2):
    """
    Builds a numeric transition matrix using a symbolic state matrix to allow/block transitions.

    Parameters:
        data         : list of numeric vectors
        state_matrix : n x n array of strings (e.g., 'A->C', 'C->B', 'ABSORBED')
        min_similarity : cosine similarity threshold
    Returns:
        matrix : n x n numeric transition matrix
    """
    n = len(data)

    # Convert state_matrix to numpy array if needed
    if not isinstance(state_matrix, np.ndarray):
        state_matrix = np.array(state_matrix)

    # Initialize matrix
    matrix = np.zeros((n, n), dtype=float)

    # Precompute cosine similarities
    cosims = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            cosims[i, j] = cosine_similarity(data[i], data[j])

    # Apply gating based on state_matrix
    for i in range(n):
        for j in range(n):
            if state_matrix[i, j] == 'ABSORBED':
                continue  # block transition
            if cosims[i, j] < min_similarity:
                continue  # below similarity threshold
            matrix[i, j] = cosims[i, j]  # allowed transition

    return matrix





In [ ]:
# PAGERANK
import numpy as np

class PageRankSolver:
    def __init__(self, adjacency_matrix,
                 damping=0.9,
                 max_iterations=1_000_000_000_000_000,
                 tol=1e-16):

        self.A = adjacency_matrix.astype(float)
        self.N = self.A.shape[0]

        assert self.A.shape[0] == self.A.shape[1], "Matrix must be square"

        self.damping = damping
        self.max_iterations = max_iterations
        self.tol = tol

        # Normalize adjacency matrix (column-stochastic)
        self.M = self._build_transition_matrix()

        # Initialize pagerank vector
        self.rank = np.ones(self.N) / self.N

    # --------------------------------------------------
    # Build transition matrix
    # --------------------------------------------------
    def _build_transition_matrix(self):
        M = self.A.copy()
        col_sums = M.sum(axis=0)

        for j in range(self.N):
            if col_sums[j] == 0:
                # Dangling node → distribute uniformly
                M[:, j] = 1.0 / self.N
            else:
                M[:, j] /= col_sums[j]

        return M

    # --------------------------------------------------
    # One PageRank iteration
    # --------------------------------------------------
    def _iterate(self):
        teleport = np.ones(self.N) / self.N
        new_rank = (
            self.damping * self.M @ self.rank
            + (1 - self.damping) * teleport
        )
        return new_rank

    # --------------------------------------------------
    # Convergence check
    # --------------------------------------------------
    def _converged(self, new_rank):
        return np.linalg.norm(new_rank - self.rank, 1) < self.tol

    # --------------------------------------------------
    # Solve PageRank
    # --------------------------------------------------
    def solve(self):
        for _ in range(self.max_iterations):
            new_rank = self._iterate()

            if self._converged(new_rank):
                break

            self.rank = new_rank

        # Normalize (numerical safety)
        self.rank /= self.rank.sum()

        return self.rank
import numpy as np

class PageRankSolver:
    def __init__(self, adjacency_matrix,
                 damping=0.9,
                 max_iterations=100000,
                 tol=1e-16):

        self.A = adjacency_matrix.astype(np.float64)
        self.N = self.A.shape[0]

        assert self.A.shape[0] == self.A.shape[1], "Matrix must be square"

        self.damping = damping
        self.max_iterations = max_iterations
        self.tol = tol

        # CHANGED: Bias is now (1 - d) instead of (1 - d) / N
        # This injects more energy into the system, preventing sum-to-one.
        self.bias = (1 - self.damping)

        # Pre-compute damped transition matrix
        self.M = self._build_transition_matrix() * self.damping

        # CHANGED: Initialize rank to 1.0 per node (Total sum = N)
        self.rank = np.ones(self.N, dtype=np.float64)

    # --------------------------------------------------
    # Build transition matrix
    # --------------------------------------------------
    def _build_transition_matrix(self):
        M = self.A.copy()
        col_sums = M.sum(axis=0)

        # Vectorized handling of Dangling Nodes
        is_dangling = np.isclose(col_sums, 0)

        # 1. Normalize valid columns
        M[:, ~is_dangling] /= col_sums[~is_dangling]

        # 2. Distribute dangling nodes uniformly
        M[:, is_dangling] = 1.0 / self.N

        return M

    # --------------------------------------------------
    # One PageRank iteration
    # --------------------------------------------------
    def _iterate(self):
        # Calculate new rank based on flow
        # We add the larger bias term here
        new_rank = (self.M @ self.rank) + self.bias

        # REMOVED: Normalization step (new_rank /= sum)
        # This allows the values to grow/float to their natural magnitude (Sum ~ N)

        return new_rank

    # --------------------------------------------------
    # Convergence check
    # --------------------------------------------------
    def _converged(self, new_rank):
        # Check convergence using L1 norm
        return np.linalg.norm(new_rank - self.rank, 1) < self.tol

    # --------------------------------------------------
    # Solve PageRank
    # --------------------------------------------------
    def solve(self):
        for _ in range(self.max_iterations):
            new_rank = self._iterate()

            if self._converged(new_rank):
                self.rank = new_rank
                break

            self.rank = new_rank

        # REMOVED: Final normalization
        # Returns raw scores. Average score will be ~1.0.
        return self.rank

import numpy as np

class PageRankSolver:
    def __init__(self, initial_matrix,
                 damping=0.85,
                 max_iterations=1000000,
                 tol=1e-16):

        self.damping = damping
        self.max_iterations = max_iterations
        self.tol = tol

        # Initialize with the first time-step matrix
        self._setup_matrix(initial_matrix)

        # Initialize rank to 1.0 (Non-sum-to-one approach)
        self.rank = np.ones(self.N, dtype=np.float64)

        # Solve initial state
        self.solve()

    # --------------------------------------------------
    # Internal: Setup Matrix & Bias
    # --------------------------------------------------
    def _setup_matrix(self, matrix):
        self.A = matrix.astype(np.float64)
        self.N = self.A.shape[0]
        assert self.A.shape[0] == self.A.shape[1], "Matrix must be square"

        # Bias term for non-sum-to-one (injecting (1-d) energy)
        self.bias = (1 - self.damping)

        # Pre-compute Transition Matrix
        M = self.A.copy()
        col_sums = M.sum(axis=0)

        # Vectorized Dangling Node Handling
        is_dangling = np.isclose(col_sums, 0)

        # Normalize valid columns
        M[:, ~is_dangling] /= col_sums[~is_dangling]

        # Dangling nodes -> uniform distribution
        M[:, is_dangling] = 1.0 / self.N

        # Apply damping once
        self.M = M * self.damping

    # --------------------------------------------------
    # DYNAMIC: Evolve the graph to the next time step
    # --------------------------------------------------
    def evolve(self, new_matrix):
        """
        Updates the graph structure and re-calculates PageRank
        using the previous rank as a 'Warm Start'.
        """
        new_N = new_matrix.shape[0]

        # 1. Handle Node Expansion (New nodes added)
        if new_N > self.N:
            # Pad the existing rank vector with the mean score (or 1.0)
            diff = new_N - self.N
            avg_score = np.mean(self.rank)
            padding = np.full(diff, avg_score, dtype=np.float64)
            self.rank = np.concatenate([self.rank, padding])

        # 2. Handle Node Contraction (Nodes removed - rare but possible)
        elif new_N < self.N:
            self.rank = self.rank[:new_N]

        # 3. Setup the new matrix
        self._setup_matrix(new_matrix)

        # 4. Solve (Starting from self.rank, not from scratch)
        return self.solve()

    # --------------------------------------------------
    # One Iteration
    # --------------------------------------------------
    def _iterate(self):
        # M @ rank + bias
        new_rank = (self.M @ self.rank) + self.bias
        return new_rank

    # --------------------------------------------------
    # Solver (with Warm Start)
    # --------------------------------------------------
    def solve(self):
        # We do NOT reset self.rank to ones here.
        # We use the existing self.rank as the "Warm Start" prediction.

        for i in range(self.max_iterations):
            new_rank = self._iterate()

            # Convergence check (L1 Norm)
            if np.linalg.norm(new_rank - self.rank, 1) < self.tol:
                self.rank = new_rank
                break # Converged

            self.rank = new_rank

        return self.rank


import numpy as np

class PageRankSolver:
    def __init__(self, adjacency_matrix,
                 damping=0.85,
                 tol=1e-12,
                 max_iterations=1_000_000_000_000_000):
        """
        Dynamic PageRank solver with step, iterate, and solve interfaces.
        """
        self.damping = damping
        self.tol = tol
        self.max_iterations = max_iterations

        self._setup_matrix(adjacency_matrix)

        # Warm-start rank (non-sum-to-one)
        self.rank = np.ones(self.N, dtype=np.float64)

    # --------------------------------------------------
    # Build transition matrix
    # --------------------------------------------------
    def _setup_matrix(self, A):
        self.A = A.astype(np.float64)
        self.N = self.A.shape[0]

        assert self.A.shape[0] == self.A.shape[1], "Matrix must be square"

        M = self.A.copy()
        col_sums = M.sum(axis=0)

        dangling = np.isclose(col_sums, 0)

        # Normalize non-dangling columns
        M[:, ~dangling] /= col_sums[~dangling]

        # Dangling nodes → uniform distribution
        M[:, dangling] = 1.0 / self.N

        # Apply damping once
        self.M = self.damping * M

        # Bias term (injects energy)
        self.bias = (1.0 - self.damping)

    # --------------------------------------------------
    # Single iteration
    # --------------------------------------------------
    def step(self):
        """
        Perform a single PageRank iteration.
        """
        return (self.M @ self.rank) + self.bias

    # --------------------------------------------------
    # Generator: yields rank one iteration at a time
    # --------------------------------------------------
    def iterate(self):
        """
        Generator yielding PageRank values at each iteration.
        """
        for _ in range(self.max_iterations):
            new_rank = self.step()
            yield new_rank.copy()

            if np.linalg.norm(new_rank - self.rank, 1) < self.tol:
                self.rank = new_rank
                return

            self.rank = new_rank

    # --------------------------------------------------
    # Solve to convergence
    # --------------------------------------------------
    def solve(self):
        """
        Run PageRank until convergence and return final rank.
        """
        for _ in range(self.max_iterations):
            new_rank = self.step()

            if np.linalg.norm(new_rank - self.rank, 1) < self.tol:
                self.rank = new_rank
                break

            self.rank = new_rank

        return self.rank

    # --------------------------------------------------
    # Dynamic graph update (warm start preserved)
    # --------------------------------------------------
    def update_graph(self, new_adjacency_matrix):
        """
        Update graph structure while keeping previous rank as warm start.
        """
        new_N = new_adjacency_matrix.shape[0]

        # Handle node addition
        if new_N > self.N:
            extra = new_N - self.N
            avg = np.mean(self.rank)
            self.rank = np.concatenate(
                [self.rank, np.full(extra, avg)]
            )

        # Handle node removal
        elif new_N < self.N:
            self.rank = self.rank[:new_N]

        self._setup_matrix(new_adjacency_matrix)
        return self.solve()


# Optimizer

In [ ]:
       #return True
# --- Inner Loop (FCM & Learning) ---
INNER_FCM_STEPS = 1       # Iterations per node simulation
INNER_LR_X = 1.0             # Learning rate for State X
INNER_LR_Y = 0.01           # Learning rate for State Y
INNER_LR_W = 1.0             # Learning rate for Weights
INNER_SVM_LR = 0.01          # SVM Learning Rate
INNER_GAMMA = 1.0            # Inter-layer neural connection strength

# --- Random Walk & Pathfinding ---
WALK_BETA = 0.3              # Novelty penalty (dampens repeated paths)
WALK_LAMBDA_COST = 1.0       # Penalty weight for cost metrics

# --- Outer Loop (Topology Optimization) ---
OUTER_GENERATIONS = 1        # Iterations per Layer
OUTER_COST_LIMIT = 1      # Normalization ceiling for scores
INTER_EDGE_THRESH = 0.02     # Min DSM weight to trigger neural link
all_results = []
gap_threshold = 0.25
for ijk in range(1,2,1):
    for jik in range(5,6):
            #Physarum_DSM_Optimizer.reset_memory()
            for kij in range(1,2):
               for lll in range(1):
                print(50*'_',ijk,50*'-',jik,50*'=')
                #===============================================================================
                OUTER_N_SIMS = 1          # More simulations to find the "Hidden Gem" paths
                WALK_MAX_STEPS = 50
                #MAX_DEPTh# Let the walker explore complex relationships deeply
                DSM_TARGET_EDGES = jik        # Allow HIGHER density (Complexity is allowed!)
                OUTER_DSM_LAYERS = kij         # Balanced hierarchy (Structure -> Systems -> Skin)
                DSM_ADDITIVE_RATE = 0.9     # Low Learning Rate: Learn slowly, don't panic.
                DSM_FEEDBACK_STR = 0.05      # Weak Feedback: Listen to problems, but don't obsess.
                WALK_TOP_K = 2               # Soft Sparsity: Consider more options per step.
                DSM_FEEDBACK_FILTER = 0.2    # Only react to major issues.
                DSM_PRUNE_THRESH = 0.02      # Keep subtle connections.
                DSM_INIT_RANGE = 0.2         # Start with a blanker slate.
                STARTING_POINT = 0           # START AT SITE ANALYSIS (Respect the Land).
                END_POINT = lll
                #=============================================================================
                # =============================================================================
                # OPTIMIZER CLASS (FIXED INITIALIZATION)
                # =============================================================================6
                class Fuzzy_Hierarchical_Multiplex:
                    def __init__(self, candidate_dims, D_graph,
                                synthetic_targets,
                                gamma_interlayer=1.0, causal_flag=False,
                                metrics=METRIC_KEYS, metric_mask=METRIC_TARGET):

                        self.candidate_dims = candidate_dims
                        self.D_graph = D_graph
                        self.synthetic_targets = synthetic_targets
                        self.causal_flag = causal_flag
                        self.best_dim_per_node = [len(t)-1 for t in synthetic_targets]
                        self.MM = metric_mask
                        self.MK = metrics
                        self.MKI = metrics + ['score']
                        self.stored_embeddings = None
                        # --- 1. NEURAL ENGINE INITIALIZATION (MOVED INSIDE __INIT__) ---
                        # This ensures self.model exists before run_inner is ever called.

                        # Flatten input dimensions: [[4], [16]...] -> [4, 16...]
                        input_dims_flat = [d[0] for d in candidate_dims]

                        # Prepare Data Tensor
                        # Ensure DATA_MATRIX is available globally
                        self.data_tensor = torch.FloatTensor(DATA_MATRIX)

                        # Initialize the Graph Network [Image of Graph Neural Network architecture]
                        self.model = GraphNetwork(
                            node_feature_dims=input_dims_flat,
                            adjacency_matrix=DSM,
                            latent_dim=32
                        )
                        # -------------------------------------------------------------

                        self.PLM = [[] for _ in range(self.D_graph)]
                        self.PLMS = [[] for _ in range(self.D_graph)]
                        self.nested_reps = [np.zeros(c[0]) for c in candidate_dims]

                        # Inter-layer setup
                        self.inter_layer = InterLayer(D_graph, max_inner_dim=max(candidate_dims), gamma=gamma_interlayer)
                        self.chosen_Gmat = np.random.uniform(0.0, 0.3, (D_graph, D_graph))
                        np.fill_diagonal(self.chosen_Gmat, 0)

                        self.l2_before, self.l2_after = [], []
                        self.max_target_len = max(len(t['target']) for t in synthetic_targets)
                        self.svm_lr = 0.01

                        self.metric_traces = {k: [] for k in metrics}
                        self.metric_traces_per_node = [{} for _ in range(self.D_graph)]

                        # DSM optimization hyperparameters
                        self.dsm_lr = 0.1
                        self.dsm_l1 = 0.02
                        self.dsm_clip = 1.0
                        self.dsm_history = []
                        self.dsm_cost_weight = 0.05
                        self.best_node_weights = {}

                    def print_dsm_basic(self):
                        D = self.D_graph
                        print("\n=== DESIGN STRUCTURE MATRIX (DSM) : Gmat ===")
                        header = "     " + " ".join([f"N{j:>4}" for j in range(D)])
                        print(header)
                        for i in range(D):
                            row = "N{:>2} | ".format(i)
                            for j in range(D):
                                row += f"{self.chosen_Gmat[i, j]:>5.2f} "
                            print(row)

                    # ---------- INNER LOOP (INTEGRATED) ----------
                    def run_inner(self, node_idx, target, D_fcm,
                  steps=INNER_FCM_STEPS, lr_x=INNER_LR_X, lr_y=INNER_LR_Y, lr_W=INNER_LR_W,
                  decorrelate_metrics=False):
                        """
                        Executes the Neural 'Brain' (GNN) to determine node states.
                        CRITICAL UPDATE: Now captures Latent Embeddings for the Topological Optimizer.
                        """

                        # =========================================================
                        # 1. EXECUTE THE NEURAL NETWORK
                        # =========================================================
                        self.model.eval()
                        with torch.no_grad():
                            derived_metrics, embeddings = self.model(self.data_tensor)

                            # [FIX] Use 'self.stored_embeddings' consistently
                            if hasattr(embeddings, 'cpu'):
                                self.stored_embeddings = embeddings.cpu().numpy()
                            else:
                                self.stored_embeddings = embeddings.numpy()

                        # =========================================================
                        # 2. EXTRACT THE RESULT FOR THE TARGET NODE
                        # =========================================================
                        # Grab the calculated score for the specific node (node_idx)
                        # We average across the batch to get a stable "System Score"
                        raw_neural_score = derived_metrics[:, node_idx].mean().item()

                        # =========================================================
                        # 3. APPLY OUTER LOOP CONTROL
                        # =========================================================
                        # Apply damping/amplification from the topology loop
                        outer_scale = getattr(self, 'best_node_weights', {}).get(node_idx, 1.0)

                        # Safety check: ensure scalar
                        if isinstance(outer_scale, (list, np.ndarray)):
                            outer_scale = 1.0

                        final_score = raw_neural_score * outer_scale

                        # =========================================================
                        # 4. FORMAT OUTPUT (Legacy Compatibility)
                        # =========================================================
                        # Map the single neural score back to the user's requested metrics
                        metric_mask = METRIC_TARGET[node_idx]
                        metric_values = {}

                        for key, mask in zip(self.MK, metric_mask):
                            if mask:
                                metric_values[key] = final_score
                                # Trace history for plotting
                                self.metric_traces[key].append((raw_neural_score, final_score))
                            else:
                                metric_values[key] = 0.0

                        metric_values['score'] = final_score

                        # Return tuple with dummy values (0.0) to match legacy function signature
                        # allowing the rest of your pipeline to run without breaking.
                        return 0.0, 0.0, 0.0, 0.0, metric_values

                    # ---------- OUTER LOOP ----------

                    # ---------- OUTER LOOP (Topology Optimization) ----------
                    def run_outer(self, outer_cost_limit=OUTER_COST_LIMIT, alpha=0.0, additive_rate=DSM_ADDITIVE_RATE):
                        """
                        OPTIMIZED: Uses Topological Information Maximization with Ragged Data Handling.
                        """
                        node_metrics_list = self.capped_node_metrics
                        D = self.D_graph

                        # =========================================================
                        # 1. STATE SPACE MAPPING (Data Prep)
                        # =========================================================

                        node_types = infer_node_types(node_metrics_list)
                        C_matrix = build_coupling_matrix(node_types)

                        print(f" [Optimizer] Architecture: {node_types}")
                        print(f" [Optimizer] Mapping State Space...")

                        # 1.0 Clean Raw Data
                        raw_data = []
                        for row in node_metrics_list:
                            if isinstance(row, dict):
                                raw_data.append(list(row.values()))
                            elif isinstance(row, (list, np.ndarray)):
                                raw_data.append(list(row))
                            else:
                                raise ValueError(f"Unexpected row type: {type(row)}")

                        if len(raw_data) != D:
                            raise ValueError(f"Length mismatch: data={len(raw_data)}, expected={D}")

                        # 1.1 NORMALIZE DATA LENGTHS (Fix for 'Inhomogeneous Shape' error)
                        # Find the maximum length among all rows to ensure rectangular matrix
                        if not raw_data:
                            max_len = 0
                        else:
                            max_len = max(len(row) for row in raw_data)

                        data_clean = []
                        for row in raw_data:
                            # Pad with zeros if shorter than max_len
                            if len(row) < max_len:
                                padded_row = row + [0.0] * (max_len - len(row))
                                data_clean.append(padded_row)
                            else:
                                data_clean.append(row)

                        # Convert to NumPy array now to fail early if still broken
                        try:
                            data_matrix = np.array(data_clean, dtype=float)
                        except ValueError as e:
                            print(f" [Optimizer] CRITICAL DATA ERROR: {e}")
                            # Emergency Fallback: Force truncation to min length
                            min_len = min(len(r) for r in raw_data)
                            data_clean = [r[:min_len] for r in raw_data]
                            data_matrix = np.array(data_clean, dtype=float)

                        # Build Transition Matrix 'T'
                        T = build_transition_matrix(data_clean, C_matrix, min_similarity=0.1)

                        print(" [Optimizer] Running Neural-Topological Optimization...")

                        # [STEP 1] Retrieve the Neural Embeddings
                        # Shape comes in as: (Batch, Nodes, Emb_Dim) -> e.g., (100, 15, 16)
                        raw_features = getattr(self, 'stored_embeddings', None)

                        if raw_features is not None:
                            # [FIX] FLATTEN BATCH DIMENSION
                            # We take the mean across axis 0 to get the "Average Semantic Meaning" of each node
                            # New Shape: (15, 16)
                            if len(raw_features.shape) == 3:
                                neural_features = np.mean(raw_features, axis=0)
                            else:
                                neural_features = raw_features

                            # [STEP 2] Generate Constraint Mask
                            # 1. Normalize (Vector Length = 1.0)
                            norms = np.linalg.norm(neural_features, axis=1, keepdims=True)
                            norms[norms == 0] = 1e-9
                            normalized_features = neural_features / norms

                            # 2. Compute "Decoded Product" (Matrix Multiplication)
                            # (15, 16) @ (16, 15) -> (15, 15)
                            # This works now because dimensions align perfectly.
                            neural_affinity = np.dot(normalized_features, normalized_features.T)

                            # 3. Create Binary Mask (Thresholding)
                            neural_mask = (neural_affinity > 0.1).astype(float)
                            np.fill_diagonal(neural_mask, 1.0)

                            print(f" [Optimizer] Generated Neural Constraint Mask (Density: {np.mean(neural_mask):.2f})")
                        else:
                            print(" [Optimizer] No neural features yet. Using Open Constraints.")
                            # Fallbacks
                            neural_features = None
                            neural_mask = np.ones((self.D_graph, self.D_graph)) # Ensure correct size D_graph

                        # [STEP 3] Initialize Optimizer

                        # [!!! FIX ENDS HERE !!!] -------------------------------------------
                        # Assuming cost_matrix is already defined for the matching problem

                                                # Solve the matching problem
                        # =========================================================
                        # 4. MATCHING (PAGERANK SOLVER)
                        # =========================================================


                        pagerank_solver = PageRankSolver(T)
                        pagerank_scores = pagerank_solver.solve()
                        print(pagerank_scores)
                        # Assume pagerank_scores is a 1D numpy array of length D

                        # Create array of node indices
                        nodes = np.arange(len(pagerank_scores))

                        # Combine nodes and scores
                        node_scores = list(zip(nodes, pagerank_scores))

                        # Sort by score descending
                        sorted_node_scores = sorted(node_scores, key=lambda x: x[1], reverse=True)

                        # Print sorted results
                        print("Node : PageRank Score")
                        for node, score in sorted_node_scores:
                            print(f"{node} : {score:.4f} : {TOUCHPOINTS[node]}")

                        N = len(pagerank_scores)
                        nodes = np.arange(N)

                        # -----------------------------
                        # Compute outgoing implication weights
                        # -----------------------------
                        implication_matrix = np.maximum(pagerank_scores[:, None] - pagerank_scores[None, :], 0)
                        outgoing_weights = implication_matrix.sum(axis=1)

                        # -----------------------------
                        # Sort nodes descending by outgoing weight
                        # -----------------------------
                        sorted_idx = np.argsort(-outgoing_weights)
                        sorted_weights = outgoing_weights[sorted_idx]

                        # -----------------------------
                        # Determine tiers dynamically
                        # -----------------------------
                        tiers = np.zeros(N, dtype=int)
                        current_tier = 1
                        tiers[sorted_idx[0]] = current_tier

                        for i in range(1, N):
                            # Start new tier if gap exceeds threshold
                            if sorted_weights[i-1] - sorted_weights[i] > gap_threshold:
                                current_tier += 1
                            tiers[sorted_idx[i]] = current_tier

                            def merge_singleton_tiers_recursively(tiers):
                                tiers = tiers.copy()

                                while True:
                                    unique_tiers, counts = np.unique(tiers, return_counts=True)
                                    tier_counts = dict(zip(unique_tiers, counts))

                                    changed = False

                                    # Forward pass
                                    for t in unique_tiers:
                                        if t == 1:
                                            continue  # cannot merge first tier backward
                                        if tier_counts[t] == 1:
                                            tiers[tiers == t] = t - 1
                                            changed = True
                                            break  # restart scanning from the beginning

                                    if not changed:
                                        break

                                    # Renumber tiers to keep them contiguous
                                    new_labels = {
                                        old: new
                                        for new, old in enumerate(np.unique(tiers), start=0)
                                    }
                                    tiers = np.array([new_labels[t] for t in tiers])

                                return tiers
                        tiers = merge_singleton_tiers_recursively(tiers)
                        # -----------------------------
                        # Combine nodes, scores, tiers
                        # -----------------------------
                        node_data = list(zip(nodes, pagerank_scores, tiers, TOUCHPOINTS))
                        node_data_sorted = sorted(node_data, key=lambda x: x[1], reverse=True)

                        # -----------------------------
                        # Print results
                        # -----------------------------
                        print(f"{'Node':<5} | {'Score':<8} | {'Tier':<4} | Name")
                        print("-" * 40)
                        for node, score, tier, name in node_data_sorted:
                            print(f"{node:<5} | {score:<8.4f} | {tier:<4} | {name}")

                        self.chosen_Gmat = neural_mask
                    #    print(type(self.chosen_Gmat))
                        # [INTEGRATION END] ---------------------------------------

                        # =========================================================
                        # 4. PRUNING & FINALIZATION
                        # =========================================================


                        # 1. Density Control (Hard Cap on Edge Count)

                        # 2. Noise Floor Pruning (Remove weak signals)
                        self.chosen_Gmat[self.chosen_Gmat < DSM_PRUNE_THRESH] = 0.0

                        # ---------------------------------------------------------
                        # INTEGRITY PRUNING: Remove Saturated Overlaps
                        # ---------------------------------------------------------
                        # Logic: If Direct Path A->B is strictly dominated by
                        # Indirect Path A->C->B, we prune A->B to force flow through C.

                        # Create a temporary binary mask of existing edges
                        nonzero_indices = np.transpose(np.nonzero(self.chosen_Gmat))

                        for u, v in nonzero_indices:
                            if u == v: continue # Ignore self-loops

                            direct_weight = self.chosen_Gmat[u, v]

                            # Check for a "better" intermediate node 'k'
                            # We use vectorization to check all 'k' at once for speed
                            vec_u_k = self.chosen_Gmat[u, :] # Flow from u -> all k
                            vec_k_v = self.chosen_Gmat[:, v] # Flow from all k -> v

                            # Indirect path weight = weight(u->k) * weight(k->v)
                            indirect_paths = vec_u_k * vec_k_v

                            # If ANY indirect path is stronger than the direct path...
                            if np.any(indirect_paths > direct_weight):
                                # ...Prune the direct path (Force flow through the hub/intermediary)
                                self.chosen_Gmat[u, v] = 0.0

                        print(f" [Optimizer] Final Graph Density: {np.count_nonzero(self.chosen_Gmat)} edges")

                        return node_metrics_list, 0.0, 0.0

                    def run(self, outer_generations=OUTER_GENERATIONS, num_dsm_layers=OUTER_DSM_LAYERS):
                        best_score = -np.inf

                        # 1. FIX INITIALIZATION:
                        # Use the random initial state as the baseline.
                        # This ensures Layer 0 captures the "Jump" from noise to structure.
                        baseline = self.chosen_Gmat.copy()
                        dsm_decomposer = DSM_Layer_Decomposer(baseline, mode='additive')
                        dsm_decomposer.current_total = baseline.copy()

                        print(f"Starting Optimization: {num_dsm_layers} Layers x {outer_generations} Gens")

                        # Define the "Building Blocks" for the 3 layers (based on your 12-15 node stack)
                        # Layer 0: Structure (Nodes 0-5), Layer 1: Systems (Nodes 6-10), Layer 2: Skin/Ops (Nodes 11-14)
                        nodes_per_layer = np.array_split(range(self.D_graph), num_dsm_layers)

                        for layer_idx in range(num_dsm_layers):
                            print(f"\n>>> COMPILING LAYER {layer_idx + 1}: {['STRUCTURE', 'SYSTEMS', 'SKIN', 'INTERIOR', 'MEP', 'FAÇADE'][layer_idx]} <<<")

                            # Determine the nodes active in this specific layer
                            active_nodes = nodes_per_layer[layer_idx]

                            for gen in range(outer_generations):
                                # 1. Inner Loop (Targeting active nodes for this layer)
                                node_metrics_list = []
                                for node_idx in range(self.D_graph):
                                    full_target = self.synthetic_targets[node_idx]['target']
                                    D_fcm = self.candidate_dims[node_idx][0]
                                    target = full_target[:D_fcm]

                                    # We simulate everything, but the "Learning" is focused on the active layer
                                    _, _, _, _, metrics = self.run_inner(node_idx, target, D_fcm)
                                    node_metrics_list.append(metrics)

                                self.capped_node_metrics = node_metrics_list

                                # 2. Outer Loop (Topology Optimization)
                                # We pass the layer_idx to run_outer if you want to adjust the WALK_TOP_K
                                # or additive_rate per layer (e.g., higher for structure, lower for skin)
                                _, capped_score, _ = self.run_outer()

                                best_score = max(best_score, capped_score)
                                print(f" [Gen {gen+1}] Score: {capped_score:.4f}", end='\r')

                            print("")

                            # 3. SNAPSHOT: The Decomposer captures the "Delta" for this layer
                            # This is where the MUX/DEMUX logic is voucher-ed.
                            dsm_decomposer.add_snapshot(self.chosen_Gmat)

                        self.dsm_layers = dsm_decomposer.layers
                        print("\nOptimization Complete. All 3 Layers Compiled.")
                        return best_score

                    # ---------- VISUALIZATIONS & ANALYSIS ----------


                    def plot_outer_fuzzy_graph(self):
                        G = nx.DiGraph()
                        for i in range(self.D_graph): G.add_node(i)
                        for i in range(self.D_graph):
                            for j in range(self.D_graph):
                                if i != j and abs(self.chosen_Gmat[i, j]) > 0.02:
                                    G.add_edge(i, j, weight=self.chosen_Gmat[i, j])

                        node_sizes = [self.best_dim_per_node[i] * 200 for i in range(self.D_graph)]
                        edge_colors = ['green' if d['weight'] > 0 else 'red' for _, _, d in G.edges(data=True)]
                        edge_widths = [abs(d['weight']) * 3 for _, _, d in G.edges(data=True)]

                        pos = nx.spring_layout(G)
                        plt.figure(figsize=(6, 6))
                        nx.draw(G, pos, node_size=node_sizes, node_color='skyblue',
                                edge_color=edge_colors, width=edge_widths, arrows=True, with_labels=True)
                        plt.title("Outer Fuzzy Multiplex Graph")
                        plt.show()



                # =============================================================================

                if __name__ == "__main__":
                    # Ensure necessary globals exist before running; otherwise this block is illustrative
                    try:
                        optimizer = Fuzzy_Hierarchical_Multiplex(
                            candidate_dims, D_graph,
                            synthetic_targets,
                            gamma_interlayer=0,
                            causal_flag=False
                        )

                        # Run Optimization
                        metrics_list = optimizer.run()

                        # Visualizations
                    # optimizer.plot_pointwise_minmax_elite()
                        #optimizer.plot_nested_activations()

                        # Compute FMT with elite bounds
                        #fmt_elite_bounds = optimizer.compute_fmt_with_elite_bounds(top_k=top_k + 10)

                        # Plot as heatmaps
                        #optimizer.plot_fmt_with_run_metrics()

                        # Compute fuzzy multiplex tensor
                        #fmt_tensor = optimizer.compute_fuzzy_metric_tensor(normalize=False)
                        #optimizer.plot_fuzzy_metric_tensor_heatmaps(fmt_tensor)

                        # Plot Contributions & Graph
                        #optimizer.plot_node_score_contribution()
                    #    optimizer.plot_outer_fuzzy_graph()

                        # Interactions
                    # tensor = optimizer.print_interactions()
                        #print("Tensor shape:", tensor.shape, '\n', tensor)

                        # Datapoints & Equations
                        #optimizer.collect_fmt_datapoints()
                        #optimizer.plot_fmt_per_datapoint()
                        #optimizer.collect_metric_traces_per_node()
                    # optimizer.plot_metric_equations_per_node()

                        # DSM Tracking Demo
                      #  dsm_tracker = DSM_Tracker(optimizer)
#
                        # Run extra DSM update
                       # optimizer.run_outer()
                    except:pass
